In [9]:
from decimal import Decimal
from pathlib import Path
from datetime import datetime, timezone
import json
import re

try:
    import boto3
    import pandas as pd
    from boto3.dynamodb.conditions import Attr, Key
    from botocore.exceptions import BotoCoreError, ClientError
except ImportError as error:
    raise SystemExit(
        "A required package is missing. Run: pip install boto3 pandas"
    ) from error

dynamodb_endpoint = "http://localhost:9000"
aws_region = "us-east-1"
table_name = "schoolpulse_student_events"
school_id = "SCH001"

pipeline_name = "SchoolPulse Python DynamoDB Pipeline Version 2"

pipeline_run_id = datetime.now(
    timezone.utc
).strftime(
    "RUN%Y%m%dT%H%M%SZ"
)

replace_previous_pipeline_events = True
include_subject_risk_events = True

expected_indexes = {
    "active_status_created_at_index",
    "event_type_created_at_index",
    "actor_created_at_index",
    "reference_created_at_index",
    "class_month_severity_index",
}

month_end_timestamp = {
    "January": "2026-01-31T18:00:00Z",
    "February": "2026-02-28T18:00:00Z",
    "March": "2026-03-31T18:00:00Z",
    "April": "2026-04-30T18:00:00Z",
}

severity_rank = {
    "High": 1,
    "Medium": 2,
    "Low": 3,
    "Information": 4,
}

print("DynamoDB Pipeline Configuration Ready")


DynamoDB Pipeline Configuration Ready


In [10]:
def normalize_column_name(value):
    text = str(value).strip()

    text = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        text,
    )

    text = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        text,
    )

    return text.strip("_").lower()


def normalize_columns(dataframe):
    result = dataframe.copy()

    result.columns = [
        normalize_column_name(column)
        for column in result.columns
    ]

    return result


def clean_text(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    return text


def clean_number(value):
    if pd.isna(value):
        return None

    return Decimal(
        str(
            round(
                float(value),
                4,
            )
        )
    )


def clean_integer(value):
    if pd.isna(value):
        return None

    return int(
        float(value)
    )


def remove_empty_values(item):
    cleaned = {}

    for key, value in item.items():
        if value is None:
            continue

        if isinstance(value, str) and not value.strip():
            continue

        cleaned[key] = value

    return cleaned


def required_columns(
    dataframe,
    columns,
    source_name,
):
    missing = sorted(
        set(columns)
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            source_name
            + " is missing these columns: "
            + ", ".join(missing)
        )


print("Data Preparation Functions Ready")

Data Preparation Functions Ready


In [11]:
def candidate_directories():
    current_directory = Path.cwd()
    home_directory = Path.home()

    return [
        current_directory / "SchoolPulse Portfolio Outputs",
        current_directory,
        home_directory / "Desktop" / "SchoolPulse Portfolio Outputs",
        home_directory / "Documents" / "SchoolPulse Portfolio Outputs",
        home_directory / "Downloads" / "SchoolPulse Portfolio Outputs",
    ]


def locate_file(file_name):
    for directory in candidate_directories():
        candidate = directory / file_name

        if candidate.exists():
            return candidate

    entered_path = input(
        "Enter the full path to "
        + file_name
        + ": "
    ).strip().strip('"')

    candidate = Path(entered_path)

    if not candidate.exists():
        raise FileNotFoundError(
            file_name
            + " was not found."
        )

    return candidate


def event_base(
    row,
    event_id,
    event_type,
    event_display_name,
    created_at,
    event_status,
    severity,
    source,
    message,
    requires_follow_up,
    reference_partition,
):
    event_month = created_at[:7]
    rank = severity_rank[severity]

    student_id = clean_text(
        row.get("student_id")
    )

    class_id = clean_text(
        row.get("class_id")
    )

    event_key = (
        created_at
        + "#"
        + event_type
        + "#"
        + event_id
    )

    item = {
        "student_id": student_id,
        "event_key": event_key,
        "event_id": event_id,
        "event_type": event_type,
        "event_display_name": event_display_name,
        "created_at": created_at,
        "event_month": event_month,
        "month_name": clean_text(
            row.get("month_name")
        ),
        "schema_version": 2,
        "school_id": school_id,
        "enrollment_id": clean_text(
            row.get("enrollment_id")
        ),
        "student_name": clean_text(
            row.get("student_name")
        ),
        "class_id": class_id,
        "class_name": clean_text(
            row.get("class_name")
        ),
        "level_name": clean_text(
            row.get("level_name")
        ),
        "class_section": clean_text(
            row.get("class_section")
        ),
        "academic_group": clean_text(
            row.get("academic_group")
        ),
        "event_status": event_status,
        "severity": severity,
        "severity_rank": rank,
        "source": source,
        "message": message,
        "requires_follow_up": requires_follow_up,
        "monthly_summary_id": clean_text(
            row.get("monthly_summary_id")
        ),
        "risk_profile_id": clean_text(
            row.get("risk_profile_id")
        ),
        "risk_level": clean_text(
            row.get("risk_level")
        ),
        "risk_reason": clean_text(
            row.get("risk_reason")
        ),
        "recommended_action": clean_text(
            row.get("recommended_action")
        ),
        "average_score": clean_number(
            row.get("average_score")
        ),
        "attendance_percentage": clean_number(
            row.get("attendance_percentage")
        ),
        "subjects_failed": clean_integer(
            row.get("subjects_failed")
        ),
        "performance_change": clean_number(
            row.get("performance_change")
        ),
        "actor_type": "SYSTEM",
        "actor_id": "PYTHON_ANALYTICS_V2",
        "actor_name": "SchoolPulse Python Analytics Pipeline",
        "pipeline_run_id": pipeline_run_id,
        "generated_by": pipeline_name,
        "deduplication_key": (
            school_id
            + "#"
            + student_id
            + "#"
            + event_type
            + "#"
            + event_id
        ),
        "event_type_partition": (
            "SCHOOL#"
            + school_id
            + "#TYPE#"
            + event_type
        ),
        "event_type_sort": (
            created_at
            + "#"
            + student_id
            + "#"
            + event_id
        ),
        "actor_partition": "ACTOR#SYSTEM#PYTHON_ANALYTICS_V2",
        "actor_sort": (
            created_at
            + "#"
            + event_type
            + "#"
            + student_id
        ),
        "reference_partition": reference_partition,
        "reference_sort": (
            created_at
            + "#"
            + event_type
            + "#"
            + event_id
        ),
        "class_month_partition": (
            "SCHOOL#"
            + school_id
            + "#CLASS#"
            + class_id
            + "#MONTH#"
            + event_month
        ),
        "class_month_sort": (
            str(rank)
            + "#"
            + created_at
            + "#"
            + student_id
            + "#"
            + event_id
        ),
    }

    if event_status in {
        "Open",
        "Pending",
        "In Progress",
    }:
        normalized_status = (
            event_status.upper()
            .replace(
                " ",
                "_",
            )
        )

        item["active_status_partition"] = (
            "SCHOOL#"
            + school_id
            + "#ACTIVE#"
            + normalized_status
        )

        item["active_status_sort"] = (
            str(rank)
            + "#"
            + created_at
            + "#"
            + student_id
            + "#"
            + event_id
        )

    return remove_empty_values(item)


print("Event Construction Foundation Ready")

Event Construction Foundation Ready


In [12]:
def build_monthly_events(monthly_data):
    events = []

    for row in monthly_data.to_dict(
        orient="records"
    ):
        month_name = clean_text(
            row.get("month_name")
        )

        created_at = month_end_timestamp.get(
            month_name
        )

        if created_at is None:
            raise ValueError(
                "No event timestamp is configured for "
                + str(month_name)
            )

        monthly_summary_id = clean_text(
            row.get("monthly_summary_id")
        )

        student_name = clean_text(
            row.get("student_name")
        )

        risk_level = clean_text(
            row.get("risk_level")
        )

        attendance_percentage = float(
            row.get("attendance_percentage")
        )

        performance_change = row.get(
            "performance_change"
        )

        if risk_level in {
            "High",
            "Medium",
        }:
            event_type = (
                risk_level.upper()
                + "_RISK_ALERT"
            )

            event_id = (
                "ORA"
                + monthly_summary_id
            )

            events.append(
                event_base(
                    row=row,
                    event_id=event_id,
                    event_type=event_type,
                    event_display_name=(
                        risk_level
                        + " Risk Alert"
                    ),
                    created_at=created_at,
                    event_status="Open",
                    severity=risk_level,
                    source=(
                        "SchoolPulse Python Risk Analysis"
                    ),
                    message=(
                        student_name
                        + " was classified as "
                        + risk_level
                        + " risk for "
                        + month_name
                        + "."
                    ),
                    requires_follow_up=True,
                    reference_partition=(
                        "MONTHLY_SUMMARY#"
                        + monthly_summary_id
                    ),
                )
            )

        if attendance_percentage < 85:
            event_id = (
                "ATA"
                + monthly_summary_id
            )

            attendance_severity = (
                "High"
                if attendance_percentage < 70
                else "Medium"
            )

            events.append(
                event_base(
                    row=row,
                    event_id=event_id,
                    event_type="ATTENDANCE_ALERT",
                    event_display_name="Attendance Alert",
                    created_at=created_at,
                    event_status="Open",
                    severity=attendance_severity,
                    source=(
                        "SchoolPulse Python Attendance Analysis"
                    ),
                    message=(
                        student_name
                        + " recorded "
                        + f"{attendance_percentage:.2f}"
                        + " percent attendance in "
                        + month_name
                        + "."
                    ),
                    requires_follow_up=True,
                    reference_partition=(
                        "MONTHLY_SUMMARY#"
                        + monthly_summary_id
                    ),
                )
            )

        is_positive_change = (
            pd.notna(performance_change)
            and float(performance_change) >= 5
        )

        if (
            risk_level == "Low"
            and is_positive_change
        ):
            event_id = (
                "PPE"
                + monthly_summary_id
            )

            events.append(
                event_base(
                    row=row,
                    event_id=event_id,
                    event_type=(
                        "POSITIVE_PERFORMANCE_EVENT"
                    ),
                    event_display_name=(
                        "Positive Performance Event"
                    ),
                    created_at=created_at,
                    event_status="Completed",
                    severity="Information",
                    source=(
                        "SchoolPulse Python Performance Analysis"
                    ),
                    message=(
                        student_name
                        + " improved by "
                        + f"{float(performance_change):.2f}"
                        + " percentage points in "
                        + month_name
                        + "."
                    ),
                    requires_follow_up=False,
                    reference_partition=(
                        "MONTHLY_SUMMARY#"
                        + monthly_summary_id
                    ),
                )
            )

    return events


print("Monthly Event Generator Ready")

Monthly Event Generator Ready


In [13]:
def build_subject_risk_events(subject_data):
    events = []

    filtered_data = subject_data[
        subject_data[
            "risk_level"
        ].isin(
            [
                "High",
                "Medium",
            ]
        )
    ]

    for row in filtered_data.to_dict(
        orient="records"
    ):
        month_name = clean_text(
            row.get("month_name")
        )

        created_at = month_end_timestamp.get(
            month_name
        )

        subject_risk_id = clean_text(
            row.get("subject_risk_id")
        )

        student_name = clean_text(
            row.get("student_name")
        )

        subject_name = clean_text(
            row.get("subject_name")
        )

        risk_level = clean_text(
            row.get("risk_level")
        )

        event_id = (
            "SRA"
            + subject_risk_id
        )

        item = event_base(
            row=row,
            event_id=event_id,
            event_type="SUBJECT_RISK_ALERT",
            event_display_name="Subject Risk Alert",
            created_at=created_at,
            event_status="Open",
            severity=risk_level,
            source=(
                "SchoolPulse Python Subject Risk Analysis"
            ),
            message=(
                student_name
                + " requires "
                + risk_level.lower()
                + " level support in "
                + subject_name
                + "."
            ),
            requires_follow_up=True,
            reference_partition=(
                "SUBJECT_RISK#"
                + subject_risk_id
            ),
        )

        item.update(
            remove_empty_values(
                {
                    "subject_risk_id": subject_risk_id,
                    "subject_id": clean_text(
                        row.get("subject_id")
                    ),
                    "subject_name": subject_name,
                    "subject_average": clean_number(
                        row.get("subject_average")
                    ),
                    "class_subject_average": clean_number(
                        row.get("class_subject_average")
                    ),
                    "performance_gap": clean_number(
                        row.get("performance_gap")
                    ),
                }
            )
        )

        events.append(item)

    return events


print("Subject Risk Event Generator Ready")

Subject Risk Event Generator Ready


In [14]:
def connect_to_dynamodb():
    resource = boto3.resource(
        "dynamodb",
        endpoint_url=dynamodb_endpoint,
        region_name=aws_region,
        aws_access_key_id="local",
        aws_secret_access_key="local",
    )

    client = resource.meta.client

    try:
        response = client.describe_table(
            TableName=table_name
        )

    except (
        BotoCoreError,
        ClientError,
    ) as error:
        raise RuntimeError(
            "The DynamoDB table could not be reached. "
            "Confirm that DynamoDB Local is running on port 9000 "
            "and that the Version 2 model has been committed."
        ) from error

    table_description = response["Table"]

    actual_indexes = {
        index["IndexName"]
        for index in table_description.get(
            "GlobalSecondaryIndexes",
            [],
        )
    }

    missing_indexes = (
        expected_indexes
        - actual_indexes
    )

    if missing_indexes:
        raise RuntimeError(
            "The committed table is missing these indexes: "
            + ", ".join(
                sorted(missing_indexes)
            )
        )

    return resource.Table(table_name)


def delete_previous_pipeline_events(table):
    deleted_count = 0

    scan_arguments = {
        "FilterExpression": Attr(
            "generated_by"
        ).eq(
            pipeline_name
        ),
        "ProjectionExpression": (
            "student_id, event_key"
        ),
    }

    while True:
        response = table.scan(
            **scan_arguments
        )

        with table.batch_writer() as batch:
            for item in response.get(
                "Items",
                [],
            ):
                batch.delete_item(
                    Key={
                        "student_id": item["student_id"],
                        "event_key": item["event_key"],
                    }
                )

                deleted_count += 1

        if "LastEvaluatedKey" not in response:
            break

        scan_arguments[
            "ExclusiveStartKey"
        ] = response[
            "LastEvaluatedKey"
        ]

    return deleted_count


def write_events(table, events):
    with table.batch_writer(
        overwrite_by_pkeys=[
            "student_id",
            "event_key",
        ]
    ) as batch:
        for item in events:
            batch.put_item(
                Item=item
            )


def count_pipeline_events(table):
    total = 0

    scan_arguments = {
        "FilterExpression": Attr(
            "generated_by"
        ).eq(
            pipeline_name
        ),
        "Select": "COUNT",
    }

    while True:
        response = table.scan(
            **scan_arguments
        )

        total += response["Count"]

        if "LastEvaluatedKey" not in response:
            break

        scan_arguments[
            "ExclusiveStartKey"
        ] = response[
            "LastEvaluatedKey"
        ]

    return total


def run_validation_queries(
    table,
    events,
):
    if not events:
        return {
            "Student Timeline Items": 0,
            "Open Work Queue Items": 0,
            "Class Month Items": 0,
        }

    first_event = events[0]

    student_response = table.query(
        KeyConditionExpression=Key(
            "student_id"
        ).eq(
            first_event["student_id"]
        )
    )

    open_response = table.query(
        IndexName=(
            "active_status_created_at_index"
        ),
        KeyConditionExpression=Key(
            "active_status_partition"
        ).eq(
            "SCHOOL#"
            + school_id
            + "#ACTIVE#OPEN"
        ),
        Limit=25,
    )

    class_response = table.query(
        IndexName=(
            "class_month_severity_index"
        ),
        KeyConditionExpression=Key(
            "class_month_partition"
        ).eq(
            first_event[
                "class_month_partition"
            ]
        ),
        Limit=25,
    )

    return {
        "Student Timeline Items": student_response[
            "Count"
        ],
        "Open Work Queue Items": open_response[
            "Count"
        ],
        "Class Month Items": class_response[
            "Count"
        ],
    }


print("DynamoDB Connection And Validation Functions Ready")

DynamoDB Connection And Validation Functions Ready


In [15]:
monthly_file = locate_file(
    "Student Monthly Analysis.csv"
)

subject_file = locate_file(
    "Student Subject Risk Analysis.csv"
)

monthly_data = normalize_columns(
    pd.read_csv(monthly_file)
)

subject_data = normalize_columns(
    pd.read_csv(subject_file)
)

required_columns(
    monthly_data,
    {
        "student_id",
        "student_name",
        "enrollment_id",
        "class_id",
        "class_name",
        "level_name",
        "class_section",
        "academic_group",
        "month_name",
        "monthly_summary_id",
        "risk_profile_id",
        "risk_level",
        "risk_reason",
        "recommended_action",
        "average_score",
        "attendance_percentage",
        "subjects_failed",
        "performance_change",
    },
    "Student Monthly Analysis",
)

required_columns(
    subject_data,
    {
        "student_id",
        "student_name",
        "enrollment_id",
        "class_id",
        "class_name",
        "level_name",
        "class_section",
        "academic_group",
        "month_name",
        "monthly_summary_id",
        "subject_risk_id",
        "subject_id",
        "subject_name",
        "risk_level",
        "risk_reason",
        "recommended_action",
        "subject_average",
        "class_subject_average",
        "performance_gap",
    },
    "Student Subject Risk Analysis",
)

monthly_events = build_monthly_events(
    monthly_data
)

subject_events = (
    build_subject_risk_events(
        subject_data
    )
    if include_subject_risk_events
    else []
)

all_events = (
    monthly_events
    + subject_events
)

duplicate_keys = len(all_events) - len(
    {
        (
            event["student_id"],
            event["event_key"],
        )
        for event in all_events
    }
)

if duplicate_keys:
    raise ValueError(
        "Duplicate DynamoDB keys were generated."
    )

print(
    "Monthly Events Generated:",
    len(monthly_events),
)

print(
    "Subject Risk Events Generated:",
    len(subject_events),
)

print(
    "Total Events Generated:",
    len(all_events),
)

print(
    "Duplicate Event Keys:",
    duplicate_keys,
)

Monthly Events Generated: 908
Subject Risk Events Generated: 5559
Total Events Generated: 6467
Duplicate Event Keys: 0


In [16]:
import boto3
from botocore.exceptions import ClientError

dynamodb = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:9000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

table_name = "schoolpulse_student_events"
index_name = "class_month_severity_index"

table_info = dynamodb.describe_table(
    TableName=table_name
)["Table"]

existing_indexes = {
    index["IndexName"]
    for index in table_info.get(
        "GlobalSecondaryIndexes",
        []
    )
}

if index_name in existing_indexes:
    print(
        "The class_month_severity_index already exists."
    )

else:
    update_arguments = {
        "TableName": table_name,
        "AttributeDefinitions": [
            {
                "AttributeName": "class_month_partition",
                "AttributeType": "S",
            },
            {
                "AttributeName": "class_month_sort",
                "AttributeType": "S",
            },
        ],
        "GlobalSecondaryIndexUpdates": [
            {
                "Create": {
                    "IndexName": index_name,
                    "KeySchema": [
                        {
                            "AttributeName": "class_month_partition",
                            "KeyType": "HASH",
                        },
                        {
                            "AttributeName": "class_month_sort",
                            "KeyType": "RANGE",
                        },
                    ],
                    "Projection": {
                        "ProjectionType": "ALL"
                    },
                }
            }
        ],
    }

    billing_mode = table_info.get(
        "BillingModeSummary",
        {}
    ).get(
        "BillingMode"
    )

    if billing_mode != "PAY_PER_REQUEST":
        update_arguments[
            "GlobalSecondaryIndexUpdates"
        ][0]["Create"]["ProvisionedThroughput"] = {
            "ReadCapacityUnits": 5,
            "WriteCapacityUnits": 5,
        }

    dynamodb.update_table(
        **update_arguments
    )

    waiter = dynamodb.get_waiter(
        "table_exists"
    )

    waiter.wait(
        TableName=table_name
    )

    print(
        "class_month_severity_index was created successfully."
    )

class_month_severity_index was created successfully.


In [18]:
table = connect_to_dynamodb()

deleted_count = 0

if replace_previous_pipeline_events:
    deleted_count = (
        delete_previous_pipeline_events(
            table
        )
    )

write_events(
    table,
    all_events,
)

stored_count = count_pipeline_events(
    table
)

validation_queries = run_validation_queries(
    table,
    all_events,
)

if stored_count != len(all_events):
    raise RuntimeError(
        "The stored event count does not match "
        "the generated event count."
    )

report = {
    "Pipeline": pipeline_name,
    "Pipeline Run ID": pipeline_run_id,
    "DynamoDB Endpoint": dynamodb_endpoint,
    "Table": table_name,
    "Monthly Events Generated": len(
        monthly_events
    ),
    "Subject Risk Events Generated": len(
        subject_events
    ),
    "Total Events Generated": len(
        all_events
    ),
    "Previous Pipeline Events Removed": deleted_count,
    "Pipeline Events Stored": stored_count,
    "Duplicate Event Keys": duplicate_keys,
    "Validation Queries": validation_queries,
    "Validation Status": "PASS",
}

report_path = (
    monthly_file.parent
    / "DynamoDB Pipeline Report.json"
)

with report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        report,
        file,
        indent=4,
    )

display(
    pd.DataFrame(
        [
            {
                "Check Name": key,
                "Result": value,
            }
            for key, value in report.items()
            if key != "Validation Queries"
        ]
    )
)

display(
    pd.DataFrame(
        [
            {
                "Validation Query": key,
                "Items Returned": value,
            }
            for key, value in validation_queries.items()
        ]
    )
)

print(
    "SchoolPulse DynamoDB Pipeline Completed Successfully"
)

,Check Name,Result
0,Pipeline,SchoolPulse Python DynamoDB Pipeline Version 2
1,Pipeline Run ID,RUN20260719T163218Z
2,DynamoDB Endpoint,http://localhost:9000
3,Table,schoolpulse_student_events
4,Monthly Events Generated,908
5,Subject Risk Events Generated,5559
6,Total Events Generated,6467
7,Previous Pipeline Events Removed,6467
8,Pipeline Events Stored,6467
9,Duplicate Event Keys,0


,Validation Query,Items Returned
0,Student Timeline Items,56
1,Open Work Queue Items,25
2,Class Month Items,25


SchoolPulse DynamoDB Pipeline Completed Successfully
